In [22]:
import sys
from pathlib import Path

In [23]:
import storysniffer
import pandas as pd
import altair as alt

In [24]:
this_dir = Path("__file__").parent.absolute()

In [25]:
sys.path.append(str(this_dir.parent / "newshomepages"))

In [26]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [27]:
analysis_dir = this_dir.parent / "_analysis"

In [28]:
df = pd.read_csv(analysis_dir / "nytimes-hyperlinks.csv")

In [41]:
df = df[(~pd.isnull(df.url)) & (~pd.isnull(df.text))].copy()

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 129498 entries, 0 to 132843
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   text             129498 non-null  object
 1   url              129498 non-null  object
 2   site_handle      129498 non-null  object
 3   item_identifier  129498 non-null  object
 4   file_timestamp   129498 non-null  object
 5   file_url         129498 non-null  object
dtypes: object(6)
memory usage: 6.9+ MB


In [43]:
df.head()

,text,url,site_handle,item_identifier,file_timestamp,file_url
0,Continue reading the main story,#after-dfp-ad-top,nytimes,nytimes-2022,2022-04-07 20:12:08,https://archive.org/download/nytimes-2022/nyti...
1,Skip to content,#site-content,nytimes,nytimes-2022,2022-04-07 20:12:08,https://archive.org/download/nytimes-2022/nyti...
2,Skip to site index,#site-index,nytimes,nytimes-2022,2022-04-07 20:12:08,https://archive.org/download/nytimes-2022/nyti...
4,U.S.,/,nytimes,nytimes-2022,2022-04-07 20:12:08,https://archive.org/download/nytimes-2022/nyti...
5,International,/international/?action=click&region=Editions&p...,nytimes,nytimes-2022,2022-04-07 20:12:08,https://archive.org/download/nytimes-2022/nyti...


In [44]:
sniffer = storysniffer.StorySniffer()

In [45]:
df['is_story'] = df.apply(lambda x: sniffer.guess(x['url'], x['text']), axis=1)

In [46]:
df.is_story.value_counts()

True     73305
False    56193
Name: is_story, dtype: int64

In [47]:
df.is_story.value_counts(normalize=True)

True     0.566071
False    0.433929
Name: is_story, dtype: float64

In [48]:
(
    df.groupby(["text", "url", "is_story"])
       .size()
       .rename("count")
       .reset_index()
       .to_csv("./nytimes-hyperlink-counts.csv", index=False)
)